# 2.1 — Baseline: SVR, DT, RF, XGBoost

Treina os quatro modelos de ML clássico do REF1 para os três outputs do processo, registrando cada run no MLflow (experimento `baseline`).

**Critério de aceitação:** R² dentro de ±0.02 dos valores publicados no test set.

## Seção 1 — Imports e configuração

In [1]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

RANDOM_STATE = 42
PROCESSED_DIR = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_0/processed")
ARTIFACTS_DIR = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_2")
MLFLOW_URI    = "file:///Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/mlruns"
EXPERIMENT    = "baseline"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", EXPERIMENT)

MLflow tracking URI: file:///Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/mlruns
Experimento: baseline


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Seção 2 — Carga dos dados pré-processados

In [2]:
X_train = np.load(PROCESSED_DIR / "X_train.npy")
X_val   = np.load(PROCESSED_DIR / "X_val.npy")
X_test  = np.load(PROCESSED_DIR / "X_test.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
y_val   = np.load(PROCESSED_DIR / "y_val.npy")
y_test  = np.load(PROCESSED_DIR / "y_test.npy")

# Scalers para desnormalização: y_orig = y_norm * scale + min
scaler_y_min   = np.load(PROCESSED_DIR / "scaler_y_min.npy")
scaler_y_scale = np.load(PROCESSED_DIR / "scaler_y_scale.npy")

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

X_train: (1536, 8) y_train: (1536, 3)
X_val:   (385, 8) y_val:   (385, 3)
X_test:  (193, 8) y_test:  (193, 3)


## Seção 3 — Definição dos grids de hiperparâmetros e modelos

In [3]:
OUTPUT_NAMES = ["M_CH3OH", "x_CH3OH", "ET"]

models = [
    (
        "SVR",
        SVR(),
        {
            "C":       [0.1, 1, 10, 100],
            "epsilon": [0.001, 0.01, 0.1],
            "kernel":  ["rbf"],
        },
    ),
    (
        "DT",
        DecisionTreeRegressor(random_state=RANDOM_STATE),
        {
            "max_depth":         [3, 5, 10, None],
            "min_samples_split": [2, 5, 10],
        },
    ),
    (
        "RF",
        RandomForestRegressor(random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100, 200],
            "max_depth":    [5, 10, None],
        },
    ),
    (
        "XGBoost",
        XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
        {
            "n_estimators":  [100, 200, 300],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1],
        },
    ),
]

def denorm_y(y_norm, output_idx):
    """Reverte a normalização MinMaxScaler para um output específico."""
    return y_norm * scaler_y_scale[output_idx] + scaler_y_min[output_idx]

def compute_metrics(y_true_norm, y_pred_norm, output_idx):
    y_true = denorm_y(y_true_norm, output_idx)
    y_pred = denorm_y(y_pred_norm, output_idx)
    return {
        "r2":  r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mse": mean_squared_error(y_true, y_pred),
    }

## Seção 4 — Loop de treinamento

In [4]:
results = []

for model_name, estimator, param_grid in models:
    for output_idx, output_name in enumerate(OUTPUT_NAMES):
        print(f"\n{'='*55}")
        print(f"  {model_name}  |  {output_name}")
        print(f"{'='*55}")

        y_tr = y_train[:, output_idx]
        y_v  = y_val[:,   output_idx]
        y_te = y_test[:,  output_idx]

        # GridSearchCV apenas no treino
        gs = GridSearchCV(
            estimator,
            param_grid,
            cv=5,
            scoring="r2",
            n_jobs=-1,
            refit=True,
        )
        gs.fit(X_train, y_tr)
        best = gs.best_estimator_
        print(f"  Melhores params: {gs.best_params_}")

        # Métricas nos 3 conjuntos (desnormalizadas)
        metrics_train = compute_metrics(y_tr, best.predict(X_train), output_idx)
        metrics_val   = compute_metrics(y_v,  best.predict(X_val),   output_idx)
        metrics_test  = compute_metrics(y_te, best.predict(X_test),  output_idx)
        print(f"  R² train={metrics_train['r2']:.4f}  val={metrics_val['r2']:.4f}  test={metrics_test['r2']:.4f}")

        # Salvar artefato .pkl
        artifact_path = ARTIFACTS_DIR / model_name / output_name
        artifact_path.mkdir(parents=True, exist_ok=True)
        pkl_path = artifact_path / "model.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump(best, f)

        # Registrar run no MLflow
        with mlflow.start_run(run_name=f"{model_name}_{output_name}"):
            mlflow.set_tag("model", model_name)
            mlflow.set_tag("output", output_name)

            mlflow.log_param("random_state", RANDOM_STATE)
            for k, v in gs.best_params_.items():
                mlflow.log_param(k, v)

            for split, m in [("train", metrics_train), ("val", metrics_val), ("test", metrics_test)]:
                mlflow.log_metric(f"{split}_r2",  m["r2"])
                mlflow.log_metric(f"{split}_mae", m["mae"])
                mlflow.log_metric(f"{split}_mse", m["mse"])

            mlflow.log_artifact(str(pkl_path), artifact_path=f"{model_name}/{output_name}")

        results.append({
            "model":       model_name,
            "output":      output_name,
            "best_params": gs.best_params_,
            **{f"train_{k}": v for k, v in metrics_train.items()},
            **{f"val_{k}":   v for k, v in metrics_val.items()},
            **{f"test_{k}":  v for k, v in metrics_test.items()},
        })

print("\nTreinamento concluído.")


  SVR  |  M_CH3OH


  Melhores params: {'C': 1, 'epsilon': 0.001, 'kernel': 'rbf'}
  R² train=0.9961  val=0.9793  test=0.9831

  SVR  |  x_CH3OH


  Melhores params: {'C': 1, 'epsilon': 0.01, 'kernel': 'rbf'}
  R² train=0.9868  val=0.9470  test=0.9531

  SVR  |  ET


  Melhores params: {'C': 10, 'epsilon': 0.001, 'kernel': 'rbf'}
  R² train=0.9998  val=0.9873  test=0.9701

  DT  |  M_CH3OH


  Melhores params: {'max_depth': None, 'min_samples_split': 10}
  R² train=0.9654  val=0.7704  test=0.7740

  DT  |  x_CH3OH
  Melhores params: {'max_depth': 10, 'min_samples_split': 10}
  R² train=0.9686  val=0.8062  test=0.7905

  DT  |  ET
  Melhores params: {'max_depth': None, 'min_samples_split': 5}
  R² train=0.9943  val=0.8577  test=0.8836

  RF  |  M_CH3OH


...

  Melhores params: {'max_depth': None, 'n_estimators': 200}
  R² train=0.9894  val=0.9315  test=0.9012

  RF  |  ET


  Melhores params: {'max_depth': None, 'n_estimators': 200}
  R² train=0.9915  val=0.9445  test=0.9240

  XGBoost  |  M_CH3OH


  Melhores params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
  R² train=0.9890  val=0.9549  test=0.9602

  XGBoost  |  x_CH3OH


  Melhores params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
  R² train=0.9995  val=0.9410  test=0.9451

  XGBoost  |  ET


  Melhores params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
  R² train=0.9995  val=0.9746  test=0.9634

Treinamento concluído.


## Seção 5 — Tabela de resultados

In [5]:
results_df = pd.DataFrame(results)

# Tabela pivotada: R² test por modelo e output
pivot = results_df.pivot(index="model", columns="output", values="test_r2").round(4)

ref1 = pd.DataFrame({
    "ET":      {"SVR": 0.903, "DT": 0.863, "RF": 0.955, "XGBoost": 0.952},
    "M_CH3OH": {"SVR": 0.907, "DT": 0.833, "RF": 0.923, "XGBoost": 0.928},
    "x_CH3OH": {"SVR": 0.864, "DT": 0.820, "RF": 0.894, "XGBoost": 0.919},
})

print("=== R² test set — obtido ===")
display(pivot)

print("\n=== R² test set — REF1 ===")
display(ref1)

delta = (pivot - ref1).round(4)
print("\n=== Δ (obtido − REF1) — critério: |Δ| ≤ 0.02 ===")
display(delta.style.applymap(lambda v: "background-color: #ffcccc" if abs(v) > 0.02 else ""))

=== R² test set — obtido ===


output,ET,M_CH3OH,x_CH3OH
model,,,
DT,0.8836,0.7740,0.7905
RF,0.9240,0.9143,0.9012
SVR,0.9701,0.9831,0.9531
XGBoost,0.9634,0.9602,0.9451



=== R² test set — REF1 ===


,ET,M_CH3OH,x_CH3OH
SVR,0.903,0.907,0.864
DT,0.863,0.833,0.820
RF,0.955,0.923,0.894
XGBoost,0.952,0.928,0.919



=== Δ (obtido − REF1) — critério: |Δ| ≤ 0.02 ===


/var/folders/yp/4mc_b2c54g7fpgchq00gm31c0000gn/T/ipykernel_14908/2894053440.py:20: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(delta.style.applymap(lambda v: "background-color: #ffcccc" if abs(v) > 0.02 else ""))


output,ET,M_CH3OH,x_CH3OH
DT,0.020600,-0.059000,-0.029500
RF,-0.031000,-0.008700,0.007200
SVR,0.067100,0.076100,0.089100
XGBoost,0.011400,0.032200,0.026100


In [6]:
# Tabela completa com todas as métricas
display(results_df[[
    "model", "output",
    "train_r2", "val_r2", "test_r2",
    "train_mae", "val_mae", "test_mae",
]].sort_values(["model", "output"]).reset_index(drop=True).round(4))

,model,output,train_r2,val_r2,test_r2,train_mae,val_mae,test_mae
0,DT,ET,0.9943,0.8577,0.8836,742.4467,4203.1330,4539.0103
1,DT,M_CH3OH,0.9654,0.7704,0.7740,369.7209,977.2285,937.5827
2,DT,x_CH3OH,0.9686,0.8062,0.7905,0.0068,0.0174,0.0189
3,RF,ET,0.9915,0.9445,0.9240,1026.6013,2569.1128,3147.0821
4,RF,M_CH3OH,0.9874,0.9076,0.9143,236.2873,626.0815,555.5247
5,RF,x_CH3OH,0.9894,0.9315,0.9012,0.0044,0.0112,0.0129
6,SVR,ET,0.9998,0.9873,0.9701,110.5960,1140.2388,1390.6690
7,SVR,M_CH3OH,0.9961,0.9793,0.9831,79.3158,281.7971,247.0333
8,SVR,x_CH3OH,0.9868,0.9470,0.9531,0.0060,0.0123,0.0119
9,XGBoost,ET,0.9995,0.9746,0.9634,281.6867,1708.6946,2230.8067


## Seção 6 — Exportar 2.1_RESULT.md

In [7]:
from datetime import datetime

TOLERANCE = 0.02

ref1 = {
    ("SVR",     "ET"):      0.903,
    ("SVR",     "M_CH3OH"): 0.907,
    ("SVR",     "x_CH3OH"): 0.864,
    ("DT",      "ET"):      0.863,
    ("DT",      "M_CH3OH"): 0.833,
    ("DT",      "x_CH3OH"): 0.820,
    ("RF",      "ET"):      0.955,
    ("RF",      "M_CH3OH"): 0.923,
    ("RF",      "x_CH3OH"): 0.894,
    ("XGBoost", "ET"):      0.952,
    ("XGBoost", "M_CH3OH"): 0.928,
    ("XGBoost", "x_CH3OH"): 0.919,
}

MODEL_ORDER  = ["SVR", "DT", "RF", "XGBoost"]
OUTPUT_ORDER = ["ET", "M_CH3OH", "x_CH3OH"]

# Buscar todas as runs do experimento e filtrar por nome em Python
all_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    order_by=["attribute.start_time DESC"],
)

valid_names = {f"{m}_{o}" for m in MODEL_ORDER for o in OUTPUT_ORDER}

# Para cada (modelo, output) manter apenas a run mais recente
run_info = {}
for _, row in all_runs.iterrows():
    name = row.get("tags.mlflow.runName", "")
    if name in valid_names and name not in run_info:
        run_info[name] = {
            "run_id": row["run_id"],
            "start":  pd.to_datetime(row["start_time"]).strftime("%Y-%m-%d %H:%M:%S"),
        }

# Indexar resultados desta execução
obtained = {
    (row["model"], row["output"]): row["test_r2"]
    for _, row in results_df.iterrows()
}

# Construir linhas da tabela de R²
rows = []
n_ok = n_fail = 0
for model in MODEL_ORDER:
    for output in OUTPUT_ORDER:
        r2_ref = ref1[(model, output)]
        r2_got = obtained[(model, output)]
        delta  = r2_got - r2_ref
        ok     = abs(delta) <= TOLERANCE
        status = "OK" if ok else "FAIL"
        sign   = "+" if delta >= 0 else ""
        if ok:
            n_ok += 1
        else:
            n_fail += 1
        rows.append(
            f"| {model:<8} | {output:<8} | {r2_ref:.4f}   | {r2_got:.4f}    "
            f"| {sign}{delta:.4f}   | {status} |"
        )

# Construir linhas da tabela de runs
run_rows = []
for model in MODEL_ORDER:
    for output in OUTPUT_ORDER:
        key  = f"{model}_{output}"
        info = run_info.get(key, {})
        rid  = info.get("run_id", "—")
        ts   = info.get("start",  "—")
        run_rows.append(f"| {model:<8} | {output:<8} | `{rid}` | {ts} |")

ts_doc  = datetime.now().strftime("%Y-%m-%d %H:%M")
total   = n_ok + n_fail
summary = "APROVADO" if n_fail == 0 else f"REPROVADO ({n_fail}/{total} fora da tolerância)"

md_lines = [
    "# 2.1 — Resultados Baseline: SVR, DT, RF, XGBoost",
    "",
    f"_Documento gerado em: {ts_doc}_",
    "",
    f"**Critério de aceitação:** |Δ| ≤ {TOLERANCE} &nbsp;·&nbsp; **{summary}**",
    "",
    "---",
    "",
    "## Runs MLflow",
    "",
    "| Modelo   | Output   | Run ID                           | Timestamp           |",
    "|----------|----------|----------------------------------|---------------------|",
    *run_rows,
    "",
    f"> Experimento: `{EXPERIMENT}` &nbsp;·&nbsp; Tracking URI: `{MLFLOW_URI}`",
    "",
    "---",
    "",
    "## R² no test set — comparação com REF1",
    "",
    "| Modelo   | Output   | R²(REF1)  | R²(obtido) | Δ         | Status |",
    "|----------|----------|-----------|------------|-----------|--------|",
    *rows,
    "",
    f"> **{n_ok}/{total}** combinações dentro da tolerância ±{TOLERANCE}.",
    "",
    "## Legenda",
    "",
    "- **OK** — |Δ| ≤ 0.02: resultado compatível com o REF1.",
    "- **FAIL** — |Δ| > 0.02: divergência além da tolerância; analisar em 2.3.",
    "- Δ positivo → implementação superou o REF1; negativo → ficou abaixo.",
]

result_path = ARTIFACTS_DIR / "2.1_RESULT_BASELINE.md"
result_path.write_text("\n".join(md_lines) + "\n", encoding="utf-8")

old_path = ARTIFACTS_DIR / "2.1_RESULT.md"
if old_path.exists():
    old_path.unlink()

print(f"Arquivo escrito: {result_path}")
print(f"Resumo: {n_ok}/{total} OK, {n_fail}/{total} FAIL")


Arquivo escrito: /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_2/2.1_RESULT_BASELINE.md
Resumo: 3/12 OK, 9/12 FAIL
